In [1]:
#=========================================================
# write_dictionaries.py
# Author: McKenna W. Stanford
# Date Created: 01/05/2024
# Utility: Writes dictionaries for each run
#=========================================================

In [1]:
#=======================================
# Imports
#=======================================
import numpy as np
import matplotlib.pyplot as plt
import glob
import xarray
import datetime
import calendar
from matplotlib.gridspec import GridSpec
import matplotlib.dates as mdates
import matplotlib
import pickle
import pandas as pd
import os
from scipy import ndimage
from scipy.ndimage import gaussian_filter
from scipy.interpolate import NearestNDInterpolator as nn
from matplotlib.patches import Rectangle
from matplotlib import cm
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
#from metpy.calc import dry_lapse, lcl, moist_lapse
#import metpy.calc as mpcalc
#from metpy.units import units
import scipy
from dask.distributed import Client
import h5py

In [2]:
#=======================================
# Diagnostics Parameters
#=======================================
space = '    '
#=======================================
# Functions
#=======================================
def print_diag(key,var):
    print(key)
    print(space,'Shape:',np.shape(var))
    print(space,'Max:',np.max(var))
    print(space,'Min:',np.min(var))
def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx],idx

# Dictionaries for 3D variables and cloud-top variables

In [3]:
def main(file):
    """
    For a single file (i.e., time), read in some variables and
    calculate water contents, integrated water contents (water paths),
    and then send variables to the 'calc_cth' function.
    
    Returns a dictionary with all desired variables.
    """

    ncfile = xarray.open_dataset(file,decode_times=False)
    pres = ncfile['pressure'].values*1.e-2 # in hPa
    rho = ncfile['rhobar'].values # in kg/m^3
    qcloud = ncfile['qc'].values # in kg/kg
    qrain = ncfile['qr'].values # in kg/kg
    qic = ncfile['qic'].values # in kg/kg
    qif = ncfile['qif'].values # in kg/kg
    qid = ncfile['qid'].values # in kg/kg
    
    qnrain = ncfile['nr'].values # in kg/kg
    qncloud = ncfile['nc'].values # in /kg
    qnic = ncfile['nic'].values # in /kg
    qnif = ncfile['nif'].values # in /kg
    qnid = ncfile['nid'].values # in /kg
    
    z = ncfile['zt'].values # in m
    w = ncfile['w_interp'].values # in m
    xtime = ncfile['time'].values[0] # seconds
    temp = ncfile['temperature'].values-273.15 # deg C
    nx = ncfile.dims['nx']
    ny = ncfile.dims['ny']
    nz = ncfile.dims['nzt']
    x = ncfile.coords['x'].values
    y = ncfile.coords['y'].values
    zw = ncfile['zw'].values
    dz = np.diff(zw)
    
    #rwc = qrain*rho*1.e3 # g/m^3
    #clwc = qcloud*rho*1.e3 # g/m^3
    #ciwc = qic*rho*1.e3 # g/m^3
    #fiwc = qif*rho*1.e3 # g/m^3
    #diwc = qid*rho*1.e3 # g/m^3


    
    dharma_dict = {'qcloud':qcloud,\
                   'qrain':qrain,\
                   'qic':qic,\
                   'qif':qif,\
                   'qid':qid,\
                   'qncloud':qncloud,\
                   'qnrain':qnrain,\
                   'qnic':qnic,\
                   'qnif':qnif,\
                   'qnid':qnid,\
                   'z':z,\
                   'w':w,\
                   'time':xtime,\
                   'temp':temp,\
                   'nx':nx,\
                   'ny':ny,\
                   'nz':nz,\
                   'x':x,\
                   'y':y,\
                   'dz':dz,\
                   'rho':rho,\
                   
                  }
    
    
    return dharma_dict

In [4]:
def get_psd_params(dharma_dict):
    """
    Calculates PSD parameters, adds them to the dictionary, and returns the dictionary.
    """
    #========================================================
    # Parameters for gamma distributions
    #========================================================
    qsmall = 1.e-14
    const_PI     = 3.14159265358979323846
    const_rhol   = 1.e3 # density of liquid water 

    #-------------------------------
    # Cloud water
    #-------------------------------
    disp = 0.3 # from input parameter file

    pgamc = 1./disp**2. - 1.
    pgamc = max(  2., pgamc )
    pgamc = min( 30., pgamc )
    gam4s1 = (pgamc+3.)*(pgamc+2.)*(pgamc+1)
    gam1c = scipy.special.gamma(pgamc + 1.)
    gam4c = scipy.special.gamma(pgamc + 4.)
    lamc_min = (pgamc+1.)/60.e-6
    lamc_max = (pgamc+1.)/1.e-6
    
    #-------------------------------
    # Rain
    #-------------------------------
    lamr_min = 1./2800.e-6
    lamr_min_hm = 1./2800.e-6
    lamr_max = 1./20.e-6
    lamr_max_hm = 1./20.e-6
    
    gamr_fix = 3.
    pgamr = gamr_fix

    gam1r = scipy.special.gamma(pgamr + 1.)
    gam4r = scipy.special.gamma(pgamr + 4.)
    
    #-------------------------------
    # Dense Ice
    #-------------------------------
    ICEMICRO_n0 = 1.e7  # non-G99 ICEMICRO only
    r_dense  = 500.e-6  # non-G99 ICEMICRO only
    m_dense  = 1.e-6    # non-G99 ICEMICRO only
    #rho_iceb = 900. # bulk ice density
    lamid_min = 1./2000.e-6 # lower limit on slope parameter for graupel
    lamid_max   = 1./20.e-6
    rho_iced = 3.*m_dense /(4.*const_PI*r_dense**3.)
    
    #-------------------------------
    # Cloud Ice
    #-------------------------------
    Dcs = 125.e-6     # threshold D for cloud ice 
    rho_icec = 500.
    pgamic = 0. # fixed value of shape factor for cloud ice
    lamic_min_d = 1./(2.*Dcs+100.e-6) # default value
    lamic_min  = lamic_min_d # when no snow, allow cloud ice to get big as snow
    lamic_max = 1./1.e-6
    mici_min = lamic_min**3./(const_PI*rho_icec) # lower limit on nic/qic
    mici_max =  lamic_max**3/(const_PI*rho_icec)# upper limit on nic/qic
    
    #-------------------------------
    # Fluffy Ice
    #-------------------------------
    m_fluffy = 1.e-9    # non-G99 ICEMICRO only
    r_fluffy = 50.e-6   # non-G99 ICEMICRO only
    lamif_max = 1./10e-6
    lamif_min = 1./2000e-6 # lower limit on slope parameter for snow
    rho_icef = 3.*m_fluffy/(4.*const_PI*r_fluffy**3.)
    #========================================================
    
    nz = dharma_dict['nz']

    dharma_dict['lam_c'] = np.zeros((dharma_dict['nx'],dharma_dict['ny'],nz))-999.
    #dharma_dict['n0_c'] = np.zeros((dharma_dict['nx'],dharma_dict['ny'],nz))-999.
    dharma_dict['log_n0_c'] = np.zeros((dharma_dict['nx'],dharma_dict['ny'],nz))-999.
    
    dharma_dict['lam_r'] = np.zeros((dharma_dict['nx'],dharma_dict['ny'],nz))-999.
    #dharma_dict['n0_r'] = np.zeros((dharma_dict['nx'],dharma_dict['ny'],nz))-999.
    dharma_dict['log_n0_r'] = np.zeros((dharma_dict['nx'],dharma_dict['ny'],nz))-999.
    
    dharma_dict['lam_ic'] = np.zeros((dharma_dict['nx'],dharma_dict['ny'],nz))-999.
    dharma_dict['log_n0_ic'] = np.zeros((dharma_dict['nx'],dharma_dict['ny'],nz))-999.
    
    dharma_dict['lam_if'] = np.zeros((dharma_dict['nx'],dharma_dict['ny'],nz))-999.
    dharma_dict['log_n0_if'] = np.zeros((dharma_dict['nx'],dharma_dict['ny'],nz))-999.    

    dharma_dict['lam_id'] = np.zeros((dharma_dict['nx'],dharma_dict['ny'],nz))-999.
    dharma_dict['log_n0_id'] = np.zeros((dharma_dict['nx'],dharma_dict['ny'],nz))-999.   
    
    for kk in range(nz):

        qcloud = dharma_dict['qcloud'][:,:,kk] # kg/kg
        qncloud = dharma_dict['qncloud'][:,:,kk] # /kg
        
        qrain = dharma_dict['qrain'][:,:,kk] # kg/kg
        qnrain = dharma_dict['qnrain'][:,:,kk] # /kg
        
        qic = dharma_dict['qic'][:,:,kk] # kg/kg
        qnic = dharma_dict['qnic'][:,:,kk] # /kg
        
        qif = dharma_dict['qif'][:,:,kk] # kg/kg
        qnif = dharma_dict['qnif'][:,:,kk] # /kg
        
        qid = dharma_dict['qid'][:,:,kk] # kg/kg
        qnid = dharma_dict['qnid'][:,:,kk] # /kg
            
        cloud_id = np.where(qcloud > qsmall)
        rain_id = np.where((qrain > qsmall) & (qnrain > qsmall) )
        ice_id = np.where(qic > qsmall)
        icef_id = np.where(qif > qsmall)
        iced_id = np.where(qid > qsmall)

        #======================================
        # Cloud
        #======================================
        lam_c = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        n0c = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        log_n0c = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.

        # Calculate Lambda
        lam_c[cloud_id] = ( const_PI*const_rhol*qncloud[cloud_id]*gam4s1 / (6.*qcloud[cloud_id]) )**(1./3.)
        
        # Lambda Limits
        min_id = np.where((lam_c < lamc_min) & (qcloud > qsmall))
        if np.size(min_id) > 0.:
            lam_c[min_id] = lamc_min
            
        max_id = np.where(lam_c > lamc_max)
        if np.size(max_id) > 0.:
            lam_c[max_id] = lamc_max

      

        # Calculate N0
        log_n0c[cloud_id] = np.log((qncloud[cloud_id])*dharma_dict['rho'][kk]) + (pgamc + 1.)*np.log(lam_c[cloud_id]) - np.log(gam1c)
        n0c[cloud_id] = np.exp(log_n0c[cloud_id])
        dharma_dict['lam_c'][:,:,kk] = lam_c
        #dharma_dict['n0_c'][:,:,kk] = n0c
        dharma_dict['log_n0_c'][:,:,kk] = log_n0c


        #======================================
        # Rain
        #======================================
        lam_r = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        n0r = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        log_n0r = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        mri = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        Dmr = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.

        lamr_min = lamr_min_hm
        lamr_max = lamr_max_hm
            

        mri_min  = lamr_min**3./(const_PI*const_rhol)
        mri_max  = lamr_max**3./(const_PI*const_rhol)

        mri[rain_id] = qnrain[rain_id]/qrain[rain_id]
        dumid = np.where( (qrain > qsmall) & (mri > mri_max))
        mri[dumid] = mri_max
        dumid = np.where( (qrain > qsmall) & (mri < mri_min))
        mri[dumid] = mri_min


        Dmr[rain_id] = ( 6. / (const_PI*const_rhol*mri[rain_id]) )**(1./3.)
        gam4s1_r = (pgamr+3.)*(pgamr+2.)*(pgamr+1.)
        lam_r[rain_id] = gam4s1_r**(1./3.)/Dmr[rain_id]

        for ii in range(len(qrain[:,0])):
            for jj in range(len(qrain[0,:])):
                if qrain[ii,jj] > qsmall:
                    lam_r[ii,jj] = max( lamr_min*(gam4s1_r/6.)**(1./3.), lam_r[ii,jj] )
                    lam_r[ii,jj] = min( lamr_max*(gam4s1_r/6.)**(1./3.), lam_r[ii,jj] )


        log_n0r[rain_id] = np.log((qnrain[rain_id])*dharma_dict['rho'][kk]) + (pgamr+1.)*np.log(lam_r[rain_id]) - np.log(gam1r)
        n0r[rain_id] = np.exp(log_n0r[rain_id])
        dharma_dict['lam_r'][:,:,kk] = lam_r
        #dharma_dict['n0_r'][:,:,kk] = n0r
        dharma_dict['log_n0_r'][:,:,kk] = log_n0r
        
        
        #======================================
        # Cloud Ice
        #======================================
        lam_ic = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        log_n0ic = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        n0ic = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        
        lam_ic[ice_id] = (const_PI*rho_icec*qnic[ice_id]/qic[ice_id])**(1./3.)
        
        # Lambda Limits
        min_id = np.where((lam_ic < lamic_min) & (qic > qsmall))
        if np.size(min_id) > 0.:
            lam_ic[min_id] = lamic_min
            
        max_id = np.where(lam_ic > lamic_max)
        if np.size(max_id) > 0.:
            lam_ic[max_id] = lamic_max
        
        n0ic[ice_id] = qnic[ice_id]*lam_ic[ice_id]
        log_n0ic[ice_id] = np.log(n0ic[ice_id])        

        dharma_dict['lam_ic'][:,:,kk] = lam_ic
        dharma_dict['log_n0_ic'][:,:,kk] = log_n0ic
        
        #======================================
        # Fluffy Ice
        #======================================
        lam_if = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        log_n0if = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        n0if = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        
        lam_if[icef_id] = (const_PI*rho_icef*qnif[icef_id]/qif[icef_id])**(1./3.)
        
        # Lambda Limits
        min_id = np.where((lam_if < lamif_min) & (qif > qsmall))
        if np.size(min_id) > 0.:
            lam_if[min_id] = lamif_min
            
        max_id = np.where(lam_if > lamif_max)
        if np.size(max_id) > 0.:
            lam_if[max_id] = lamif_max
        
        n0if[icef_id] = qnif[icef_id]*lam_if[icef_id]
        log_n0if[icef_id] = np.log(n0if[icef_id]) 

        dharma_dict['lam_if'][:,:,kk] = lam_if
        dharma_dict['log_n0_if'][:,:,kk] = log_n0if
        #print(kk)
        #print(np.max(lam_if))
        #print(np.unique(lam_if))
        #print('')
        
        #======================================
        # Dense Ice
        #======================================
        lam_id = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        log_n0id = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        n0id = np.zeros((dharma_dict['nx'],dharma_dict['ny']))-999.
        
        lam_id[iced_id] = (const_PI*rho_iced*qnid[iced_id]/qid[iced_id])**(1./3.)
        
        # Lambda Limits
        min_id = np.where((lam_id < lamid_min) & (qid > qsmall))
        if np.size(min_id) > 0.:
            lam_id[min_id] = lamid_min
            
        max_id = np.where(lam_id > lamid_max)
        if np.size(max_id) > 0.:
            lam_id[max_id] = lamid_max
        
        n0id[iced_id] = qnid[iced_id]*lam_id[iced_id]
        log_n0id[iced_id] = np.log(n0id[iced_id]) 

        dharma_dict['lam_id'][:,:,kk] = lam_id
        dharma_dict['log_n0_id'][:,:,kk] = log_n0id
                
    return dharma_dict

In [5]:
def process_3d_files(file_3d):

    dharma_dict = main(file_3d)
    dharma_dict = get_psd_params(dharma_dict)
    #print(np.unique(dharma_dict['lam_if']))

    if True:
        dsd_params_dict = {'time':dharma_dict['time'],\
                     'z':dharma_dict['z'],\
                     'x':dharma_dict['x'],\
                     'y':dharma_dict['y'],\
                     'lam_r':dharma_dict['lam_r'],\
                     'lam_c':dharma_dict['lam_c'],\
                     'lam_ic':dharma_dict['lam_ic'],\
                     'lam_if':dharma_dict['lam_if'],\
                     'lam_id':dharma_dict['lam_id'],\
                     'log_n0_r':dharma_dict['log_n0_r'],\
                     'log_n0_c':dharma_dict['log_n0_c'],\
                     'log_n0_ic':dharma_dict['log_n0_ic'],\
                     'log_n0_if':dharma_dict['log_n0_if'],\
                     'log_n0_id':dharma_dict['log_n0_id'],\
                    }
        

    if True:
        time_str = str(int(dharma_dict['time']))
        save_path = '/pscratch/sd/m/mckenna/dharma_3d/psd_params/'
        file_name = save_path+sim_name+'_psd_params_dict_{}.p'.format(time_str)
        pickle.dump(dsd_params_dict,open(file_name,"wb")) 
    
    #return dharma_dict

In [6]:
#=======================================
# Get files
#=======================================
sim_name = 'sip_10x_bulk_ice_ABIFM_hm'
path_3d = '/pscratch/sd/m/mckenna/dharma_3d/'
files_3d = sorted(glob.glob(path_3d+sim_name+'/dharma_3d_'+sim_name+'*.nc'))
print(files_3d)
print(len(files_3d))

['/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_021600.nc', '/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_022200.nc', '/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_022800.nc', '/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_023400.nc', '/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_024000.nc', '/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_024600.nc', '/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_025200.nc', '/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_025800.nc', '/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_026400.nc', '/pscratch/sd/m/mckenna/dha

In [7]:
for file in files_3d:
    print(file)
    process_3d_files(file)
    #print(aaaaa)

/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_021600.nc


/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/tmp/ipykernel_1336561/2310327257.py:29: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  nx = ncfile.dims['nx']
/tmp/ipykernel_1336561/2310327257.py:30: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  ny = ncfile.dims['ny']
/tmp/ipykernel_1336561/2310327257.py:31: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `D

/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_022200.nc
/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_022800.nc
/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_023400.nc
/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_024000.nc
/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_024600.nc
/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_025200.nc
/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_025800.nc
/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_026400.nc
/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM_hm/dharma_3d_sip_10x_bulk_ice_ABIFM_hm_027000.nc
/pscratch/sd/m/mckenna/dharma_3d/sip_10x_bulk_ice_ABIFM

In [11]:
print('done')

done
